In [7]:
import pandas as pd
import ast  
import os


file_path = 'endomondoHR_proper.json'
output_file = 'endomondo_summaries.csv' 
summaries = []

print("Starting FULL dataset processing and summary extraction")


# DATA EXTRACTION & EXTRACTION LOOP
with open(file_path, 'r') as f:
    for i, line in enumerate(f):
        try:
            # Safely parse the Python dictionary string
            row = ast.literal_eval(line.strip())
            
            # Extract high-frequency sensor streams (default to empty list if missing)
            hr_raw = row.get('heart_rate', [])
            ts = row.get('timestamp', [])
            speed_raw = row.get('speed', [])
            lat = row.get('latitude', [])
            lon = row.get('longitude', [])
            
            # The raw data can contain internal 'None' values within arrays (e.g., [120, None, 122]).
            # We filter these out using list comprehensions to prevent calculation crashes.
            hr = [x for x in hr_raw if x is not None] if hr_raw else []
            speed = [x for x in speed_raw if x is not None] if speed_raw else []
            

            summary = {
                'userId': row.get('userId'),
                'sport': row.get('sport'),
                
                # Math calculations are now protected against missing data errors
                'avg_hr': sum(hr) / len(hr) if len(hr) > 0 else None,
                'avg_speed': sum(speed) / len(speed) if len(speed) > 0 else None,
                
                # Time anchors for recency/frequency modeling
                'start_time': pd.to_datetime(ts[0], unit='s') if (ts and len(ts) > 0) else None,
                'duration_min': (ts[-1] - ts[0]) / 60 if (ts and len(ts) > 1) else 0,
                
                # Spatial coordinates
                'start_lat': lat[0] if (lat and len(lat) > 0) else None,
                'start_lon': lon[0] if (lon and len(lon) > 0) else None
            }
            
            summaries.append(summary)
            
            # Progress tracker for heavy datasets
            if i % 50000 == 0 and i > 0:
                print(f" -> {i} rows parsed successfully...")
                
        except Exception as e:
            # Captures unexpected row anomalies without bringing down the data pipeline
            continue 

print("\nCompiling extracted list into a pandas DataFrame...")

if len(summaries) == 0:
    print("\n[ERROR] The summaries list is still empty. Check raw file permissions.")
else:
    df_final = pd.DataFrame(summaries)

    # Integrity drop: Remove observations lacking user metrics or tracking timelines
    df_final = df_final.dropna(subset=['userId', 'start_time'])

    # Write out final compressed tabular dataset
    df_final.to_csv(output_file, index=False)

    print(f"\nSUCCESS: Processed {len(df_final)} valid user workouts.")
    print(f"Aggregated summary dataset saved to: {os.path.abspath(output_file)}")

Starting FULL dataset processing and summary extraction
 -> 50000 rows parsed successfully...
 -> 100000 rows parsed successfully...
 -> 150000 rows parsed successfully...

Compiling extracted list into a pandas DataFrame...

SUCCESS: Processed 167783 valid user workouts.
Aggregated summary dataset saved to: /Users/lmgc/Library/CloudStorage/OneDrive-Personal/Data Science/Final Project/11endomondo_summaries.csv
